# 01 · Items: every core PPD dataflow

An **item** is one core dataflow: `(data_type, purpose, action, source, handling_context)` using only the core
`ppd:` terms of draft-dsmullen-ppd-taxonomy-05 §6. That is 7 × 8 × 4 × 7 × 4 = **6,272 items**. They are the
columns of the household matrix.

Qualifiers (retention, processing boundary) are **not** extra items: they are modifiers on a household's answer
(*allow if not kept*, *allow if processed in the home*), so the matrix stays 6,272 wide.

Each item gets flags, never filters:
- `leaves_home`: handling context is not `ppd:householdContext` (the app uses the same test);
- `in_corpus`: one of the researched devices declares it (`data/corpus/flows.csv`, from `ppd-preference-app`);
- `plausible`: consistent with the term definitions (only rule: `ppd:publicAudience` is a disclosure context, so only `transfer` fits; taxonomy §6.6, §6.4);
- `limits`: which qualifier values can refine a decision: `retention=ppd:ephemeral` for every action (§6.7.1); `processing_boundary` in-home / on-device only for `use` and `inference` (§6.7.2, protocol §7.12).

Why one column per 5-field combination: a PPD policy rule has one value per core dimension, and a policy should not hold two rules with the same core dimensions but different effects (protocol-04 §7.11). Qualifiers refine a rule (§7.12), so they live in the cell, not in extra columns.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
import pandas as pd
from ppd_presets.items import build_items, load_corpus
from ppd_presets.taxonomy import FIELD_TERMS, short

corpus = load_corpus(ROOT / "data/corpus/flows.csv")
items = build_items(corpus)
print(f"{len(items):,} items; {items.in_corpus.sum()} declared by researched devices; {items.plausible.sum():,} plausible")
pd.Series({f: len(t) for f, t in FIELD_TERMS.items()}, name="core terms")

6,272 items; 52 declared by researched devices; 5,096 plausible


data_type           7
purpose             8
action              4
source              7
handling_context    4
Name: core terms, dtype: int64

## The researched devices

88 flows from 4 devices, using 52 distinct items. Several flows share an item (the same dataflow in different phases).

In [2]:
summary = corpus.groupby("device_name").agg(flows=("flow_id", "size"))
summary["distinct items"] = corpus.groupby("device_name").apply(lambda g: g[["data_type","purpose","action","source","handling_context"]].drop_duplicates().shape[0])
summary["with a qualifier"] = corpus.groupby("device_name").apply(lambda g: (g.retention.notna() | g.processing_boundary.notna()).sum())
summary

,flows,distinct items,with a qualifier
device_name,,,
Aqara Thermostat Hub W200,22,17,1
LG OLED65C6PUA C6,19,18,1
Level Lock,21,19,1
Reolink E1 Zoom,26,17,6


In [3]:
# Which core terms the real devices actually use, per field.
for f in ["data_type", "purpose", "action", "source", "handling_context"]:
    used = corpus[f].value_counts()
    unused = [short(t) for t in FIELD_TERMS[f] if t not in used.index]
    print(f"{f:17} used: " + ", ".join(f"{short(k)} ({v})" for k, v in used.items()) + (f"   | never: {', '.join(unused)}" if unused else ""))

data_type         used: deviceAndNetworkData (25), contentData (19), sensorData (14), identifierData (11), usageData (10), locationData (6), profileData (3)
purpose           used: coreFunctionality (51), security (11), diagnosticsAndMaintenance (8), advertisingAndMarketing (7), analyticsAndImprovement (7), personalization (2), communicationsAndNotifications (2)   | never: legalCompliance
action            used: collection (36), transfer (31), use (11), inference (10)
source            used: participantGenerated (24), userProvided (23), derivedFromPriorData (18), participantObserved (13), householdProvided (4), vendorProvided (4), thirdPartyProvided (2)
handling_context  used: vendorContext (43), householdContext (38), thirdPartyContext (7)   | never: publicAudience


## Flags

In [4]:
pd.DataFrame({
    "items": [len(items)],
    "leave the home": [items.leaves_home.sum()],
    "allow a processing-boundary limit": [items.limits.str.contains("processing_boundary").sum()],
    "plausible": [items.plausible.sum()],
    "in corpus": [items.in_corpus.sum()],
    "in corpus but implausible": [(items.in_corpus & ~items.plausible).sum()],
}).T.rename(columns={0: "count"})

,count
items,6272
leave the home,4704
allow a processing-boundary limit,3136
plausible,5096
in corpus,52
in corpus but implausible,0


In [5]:
items.loc[~items.plausible, "implausible_reason"].value_counts()

implausible_reason
public audience is a disclosure context; only transfer discloses    1176
Name: count, dtype: int64

In [6]:
items[items.in_corpus].sort_values("corpus_count", ascending=False).head(10)[
    ["data_type", "purpose", "action", "source", "handling_context", "corpus_count", "corpus_devices"]]

,data_type,purpose,action,source,handling_context,corpus_count,corpus_devices
1,ppd:identifierData,ppd:coreFunctionality,ppd:collection,ppd:userProvided,ppd:vendorContext,6,aqara-w200|level-lock|lg-oled65c6pua-c6|reolin...
4480,ppd:deviceAndNetworkData,ppd:coreFunctionality,ppd:collection,ppd:userProvided,ppd:householdContext,6,aqara-w200|level-lock|lg-oled65c6pua-c6|reolin...
4993,ppd:deviceAndNetworkData,ppd:diagnosticsAndMaintenance,ppd:transfer,ppd:participantGenerated,ppd:vendorContext,5,aqara-w200|level-lock|lg-oled65c6pua-c6
2236,ppd:sensorData,ppd:security,ppd:inference,ppd:derivedFromPriorData,ppd:householdContext,4,reolink-e1-zoom
2753,ppd:contentData,ppd:coreFunctionality,ppd:transfer,ppd:participantGenerated,ppd:vendorContext,3,reolink-e1-zoom
6001,ppd:usageData,ppd:analyticsAndImprovement,ppd:transfer,ppd:participantGenerated,ppd:vendorContext,3,aqara-w200|lg-oled65c6pua-c6|reolink-e1-zoom
1796,ppd:sensorData,ppd:coreFunctionality,ppd:collection,ppd:participantObserved,ppd:householdContext,3,aqara-w200|lg-oled65c6pua-c6|reolink-e1-zoom
3028,ppd:contentData,ppd:security,ppd:collection,ppd:participantObserved,ppd:householdContext,2,reolink-e1-zoom
1956,ppd:sensorData,ppd:personalization,ppd:use,ppd:derivedFromPriorData,ppd:householdContext,2,aqara-w200
513,ppd:identifierData,ppd:diagnosticsAndMaintenance,ppd:transfer,ppd:participantGenerated,ppd:vendorContext,2,aqara-w200|level-lock


In [7]:
DATA.mkdir(exist_ok=True)
items.to_csv(DATA / "items.csv", index=False)
print("wrote", (DATA / "items.csv").relative_to(ROOT))

wrote data/items.csv
